# 08 Refactoring Code

So far, the project was intentionally notebook-first. That made the modeling workflow easy to inspect step by step.

Now we start moving repeated logic into `src/`. The first candidate is evaluation. We used the same metrics and business KPI in several notebooks. If this logic is copied and edited independently, results can drift apart even when the model is unchanged.

## 1. Why Refactor Now?

Refactoring does not mean changing the model. It means changing the structure of the code while preserving behavior.

The concrete problem here is duplicated project logic:

- Notebook 02 cleans the raw Telco data.
- Notebook 03 creates features and preprocessing outputs.
- Notebook 04 evaluates baseline models.
- Notebook 05 evaluates tuned XGBoost models.
- Notebook 07 logs metrics to MLflow.

These steps should not be redefined differently in every notebook. Otherwise, small differences in assumptions, thresholds, feature definitions, or business KPIs can make results unreliable.

## 2. Target Structure

We start with a small but meaningful refactoring step:

```text
src/
  dataset.py          # loading and cleaning raw data
  features.py         # business features, train/test split, preprocessing
  evaluation.py       # model metrics and business KPIs
  modeling/
    train.py          # model construction, fitting, model persistence
    predict.py        # prediction helper for fitted classifiers
```

The modules are still specific to our current Telco Churn project. That is intentional. We refactor what we actually need instead of building a generic framework too early.

In [1]:
from pathlib import Path
import sys

import joblib
import pandas as pd


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "data").exists() and (path / "src").exists():
            return path
    raise FileNotFoundError("Could not find project root. Please run this notebook inside the project repository.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import clean_telco_data, load_raw_data
from src.evaluation import DEFAULT_BUSINESS_PARAMS, evaluate_classifier
from src.features import (
    create_business_features,
    create_train_test_split,
    preprocess_train_test,
    split_features_and_target,
)
from src.modeling.predict import predict
from src.modeling.train import build_baseline_models, load_model

2026-06-18 11:08:40.858 | INFO     | src.config:<module>:11 - PROJ_ROOT path is: /home/cbaldermann/Projekte/ADS_II_MLOPS/2026-06-ads-II-master


In [2]:
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Telco-Customer-Churn.csv"
INTERIM_DATA_PATH = PROJECT_ROOT / "data" / "interim" / "telco_churn_cleaned.csv"
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_RESULTS_DIR = PROJECT_ROOT / "reports" / "model_results"
MODELS_DIR = PROJECT_ROOT / "models"

print(PROJECT_ROOT)

/home/cbaldermann/Projekte/ADS_II_MLOPS/2026-06-ads-II-master


## 3. Check Refactored Data Cleaning

The cleaning function now lives in `src.dataset`. We reload the raw data, clean it again in memory, and compare it with the cleaned file produced by Notebook 02.

In [3]:
raw_df = load_raw_data(RAW_DATA_PATH)
cleaned_from_src = clean_telco_data(raw_df)
cleaned_saved = pd.read_csv(INTERIM_DATA_PATH)

print("Cleaned from src:", cleaned_from_src.shape)
print("Saved interim data:", cleaned_saved.shape)

if cleaned_from_src.shape != cleaned_saved.shape:
    raise AssertionError("Refactored cleaning output has a different shape.")

if cleaned_from_src["ChurnBinary"].tolist() != cleaned_saved["ChurnBinary"].tolist():
    raise AssertionError("Refactored cleaning changed the target values.")

print("Data cleaning check passed.")

Cleaned from src: (7043, 22)
Saved interim data: (7043, 22)
Data cleaning check passed.


## 4. Check Refactored Feature Engineering

The feature logic now lives in `src.features`. We recreate the business features, train/test split, and preprocessing output. Then we compare the result with the saved processed data from Notebook 03.

In [4]:
featured_from_src = create_business_features(cleaned_from_src)
X, y = split_features_and_target(featured_from_src)
X_train, X_test, y_train, y_test = create_train_test_split(X, y)

X_train_processed, X_test_processed, feature_names, preprocessor = preprocess_train_test(
    X_train,
    X_test,
)

saved_X_train = pd.read_csv(DATA_DIR / "X_train_processed.csv")
saved_X_test = pd.read_csv(DATA_DIR / "X_test_processed.csv")
saved_y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze("columns")
saved_y_test = pd.read_csv(DATA_DIR / "y_test.csv").squeeze("columns")

print("Processed train from src:", X_train_processed.shape)
print("Saved processed train:", saved_X_train.shape)

if X_train_processed.reset_index(drop=True).round(10).equals(saved_X_train.round(10)) is False:
    raise AssertionError("Refactored feature preprocessing changed X_train_processed.")
if X_test_processed.reset_index(drop=True).round(10).equals(saved_X_test.round(10)) is False:
    raise AssertionError("Refactored feature preprocessing changed X_test_processed.")
if y_train.reset_index(drop=True).equals(saved_y_train) is False:
    raise AssertionError("Refactored split changed y_train.")
if y_test.reset_index(drop=True).equals(saved_y_test) is False:
    raise AssertionError("Refactored split changed y_test.")

print("Feature engineering check passed.")

Processed train from src: (5634, 53)
Saved processed train: (5634, 53)
Feature engineering check passed.


## 5. Check Refactored Modeling Helpers

The baseline model definitions now live in `src.modeling.train`. This keeps model classes and default parameters in one place.

In [5]:
baseline_models = build_baseline_models()
baseline_models

{'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
 'Decision Tree': DecisionTreeClassifier(random_state=42),
 'XGBoost': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric='logloss',
               feature_types=None, feature_weights=None, gamma=None,
               grow_policy=None, importance_type=None,
               interaction_constraints=None, learning_rate=None, max_bin=None,
               max_cat_threshold=None, max_cat_to_onehot=None,
               max_delta_step=None, max_depth=None, max_leaves=None,
               min_child_weight=None, missing=nan, monotone_constraints=None,
               multi_strategy=None, n_estimators=None, n_jobs=-1,
               num_parallel_tree=None, ...)}

## 6. Load the Same Data and Model

To check that refactoring preserved behavior, we reuse the processed data and the tuned XGBoost candidate from the previous notebooks.

In [ ]:
X_train = pd.read_csv(DATA_DIR / "X_train_processed.csv")
X_test = pd.read_csv(DATA_DIR / "X_test_processed.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze("columns")
y_test = pd.read_csv(DATA_DIR / "y_test.csv").squeeze("columns")

candidate_models = sorted(MODELS_DIR.glob("*_XGBClassifier_tuned_candidate.joblib"))
if not candidate_models:
    raise FileNotFoundError("No tuned XGBoost candidate found in models/. Please run Notebook 05 first.")

model_path = candidate_models[-1]
model = load_model(model_path)

print(model_path.name)

20260616_161458_XGBClassifier_tuned_candidate.joblib


## 7. Evaluate with the Reused Function

The notebook no longer defines its own evaluation function. It imports the shared function from `src.evaluation`.

In [7]:
result = evaluate_classifier(
    model=model,
    model_name="XGBoost Bayesian Search",
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    threshold=0.5,
    business_params=DEFAULT_BUSINESS_PARAMS,
)

summary_columns = [
    "model_name",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "brier_score",
    "train_f1",
    "test_f1",
    "generalization_gap",
    "contacted_customers",
    "expected_net_value",
    "realized_net_value_for_backtest",
]

refactored_summary = pd.DataFrame([{column: result[column] for column in summary_columns}])
refactored_summary.round(4)

,model_name,accuracy,precision,recall,f1,roc_auc,brier_score,train_f1,test_f1,generalization_gap,contacted_customers,expected_net_value,realized_net_value_for_backtest
0,XGBoost Bayesian Search,0.8077,0.6782,0.5241,0.5913,0.8467,0.135,0.6128,0.5913,0.0216,228,2789.7773,3000.0


The prediction helper in `src.modeling.predict` creates the same kind of probability-based prediction output that we will later need for serving and batch inference.

In [8]:
prediction_preview = predict(model, X_test.head(), threshold=0.5)
prediction_preview

,churn_probability,churn_prediction
0,0.035957,0
1,0.725525,1
2,0.075791,0
3,0.375989,0
4,0.024084,0


## 8. Check Against the Existing Saved Results

A refactoring step should not silently change the result. We compare the new evaluation output with the values saved in Notebook 05.

In [9]:
previous_results = pd.read_csv(MODEL_RESULTS_DIR / "05_xgboost_tuning_metrics.csv")
previous_bayes = previous_results[previous_results["model_name"] == "XGBoost Bayesian Search"].iloc[0]

comparison_columns = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "brier_score",
    "train_f1",
    "test_f1",
    "generalization_gap",
    "expected_net_value",
    "realized_net_value_for_backtest",
]

comparison = pd.DataFrame(
    {
        "metric": comparison_columns,
        "previous_notebook_result": [previous_bayes[column] for column in comparison_columns],
        "refactored_function_result": [result[column] for column in comparison_columns],
    }
)
comparison["absolute_difference"] = (
    comparison["previous_notebook_result"] - comparison["refactored_function_result"]
).abs()

comparison

,metric,previous_notebook_result,refactored_function_result,absolute_difference
0,accuracy,0.807665,0.807665,0.000000e+00
1,precision,0.678201,0.678201,0.000000e+00
2,recall,0.524064,0.524064,0.000000e+00
3,f1,0.591252,0.591252,0.000000e+00
4,roc_auc,0.846698,0.846698,0.000000e+00
5,brier_score,0.134951,0.134951,5.551115e-17
6,train_f1,0.612807,0.612807,0.000000e+00
7,test_f1,0.591252,0.591252,0.000000e+00
8,generalization_gap,0.021555,0.021555,7.632783e-17
9,expected_net_value,2789.777344,2789.777344,0.000000e+00


In [10]:
max_difference = comparison["absolute_difference"].max()
print(f"Maximum absolute difference: {max_difference:.12f}")

if max_difference > 1e-9:
    raise AssertionError("Refactored evaluation does not match the saved notebook result.")

print("Refactoring check passed.")

Maximum absolute difference: 0.000000000000
Refactoring check passed.


## 9. What Changed?

The behavior should be the same. The structure changed:

Before refactoring, each notebook could define its own evaluation logic. That is flexible, but risky once several notebooks, scripts, or team members depend on the same definition.

After refactoring, the evaluation function has one source of truth in `src/evaluation.py`. Future notebooks and training scripts should import it instead of redefining the same logic.

## 10. Next Refactoring Candidates

The same logic applies to other parts of the project:

- MLflow logging
- hyperparameter search spaces
- report generation
- command-line pipeline execution

We should not move everything at once. A practical refactoring sequence starts with repeated, stable logic. Evaluation is a good first step because it directly affects model comparison and business interpretation.